# SCM setup and verification

This notebook creates a private Python environment for the course, installs the SCM from this repository, registers a Jupyter kernel, and verifies the installation. Run it from the repository root.

A notebook cannot replace the Python process that is currently running it. After the installation cell finishes, select **Kernel → Change Kernel → Python (GCM SCM)**, restart, and continue with the verification cells.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while root != root.parent and not (root / 'pyproject.toml').exists():
    root = root.parent

if not (root / 'pyproject.toml').exists():
    raise FileNotFoundError('Open this notebook from inside the GCM repository.')
if sys.version_info < (3, 11):
    raise RuntimeError('Python 3.11 or newer is required.')

print('repository:', root)
print('bootstrap python:', sys.executable)
print('version:', sys.version.split()[0])

## Create the environment

This cell creates `.venv` in the repository and installs the model, plotting tools, tests, Jupyter kernel support, and the local source in editable mode. It may take several minutes because PyTorch is a large dependency. Re-running the cell is safe.

In [ ]:
import os
import subprocess

venv = root / '.venv'
python = venv / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')

if not python.exists():
    subprocess.run([sys.executable, '-m', 'venv', str(venv)], check=True)

subprocess.run([str(python), '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([str(python), '-m', 'pip', 'install', '-e', f'{root}[dev]', 'ipykernel'], check=True)
subprocess.run([
    str(python), '-m', 'ipykernel', 'install', '--user',
    '--name', 'gcm-scm', '--display-name', 'Python (GCM SCM)',
], check=True)

print('environment ready:', venv)
print('now change this notebook kernel to Python (GCM SCM) and restart')

## Verify the selected kernel

Run the cells below only after changing to **Python (GCM SCM)**.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import scm

print('kernel python:', sys.executable)
print('scm source:', Path(scm.__file__).resolve())
print('numpy:', np.__version__)
print('pytorch:', torch.__version__)
print('accelerator available:', torch.cuda.is_available())

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider'],
    cwd=root,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('SCM tests did not pass')

Setup is complete when the test cell reports all tests passing. Continue with `02_experiments.ipynb`.